# 06｜起站 ID 缺漏診斷
以一月借出事件為分母。日型沿用 04 冊，1/1、1/9、1/20 單列。缺漏率描述可歸站資料的覆蓋，原因仍待核對。完整明細可用 display(hourly) 檢視。

In [1]:
import sqlite3
from zipfile import ZipFile
import pandas as pd

with ZipFile('../data/raw/202501-capitalbikeshare-tripdata.zip') as z:
    with z.open('202501-capitalbikeshare-tripdata.csv') as f:
        rides = pd.read_csv(f, usecols=['started_at','rideable_type','start_station_id'], dtype='string')
time = pd.to_datetime(rides.started_at, format='mixed', errors='coerce')
keep = time.ge('2025-01-01') & time.lt('2025-02-01')
t = time[keep]
day_type = (t.dt.dayofweek >= 5).map({True:'weekend', False:'weekday'})
day_type = day_type.mask(t.dt.day.isin([1,9,20]), 'special')
events = pd.DataFrame({'bike': rides.rideable_type[keep], 'day_type':day_type,
                       'hour': t.dt.hour, 'missing': rides.start_station_id[keep].fillna('').str.strip().eq('').astype(int)})
with sqlite3.connect(':memory:') as db:
    events.to_sql('events', db, index=False)
    def summary(keys):
        return pd.read_sql_query(f'''
            SELECT {keys}, COUNT(*) AS trips, SUM(missing) AS missing_ids,
                   ROUND(100.0 * SUM(missing) / COUNT(*), 2) AS missing_pct
            FROM events GROUP BY {keys} ORDER BY {keys}
        ''', db)
    by_bike = summary('bike')
    by_day = summary('day_type')
    hourly = summary('bike, day_type, hour')
assert hourly.trips.sum() == len(events) == 285695
assert hourly.missing_ids.sum() == events.missing.sum() == 88110
print(by_bike.to_string(index=False))
print(by_day.to_string(index=False))
print(hourly.loc[hourly.day_type.eq('weekday') & hourly.hour.isin([7,8,9,16,17,18])].to_string(index=False))


         bike  trips  missing_ids  missing_pct
 classic_bike  78942            0         0.00
electric_bike 206753        88110        42.62
day_type  trips  missing_ids  missing_pct
 special  20812         7308        35.11
 weekday 197145        60372        30.62
 weekend  67738        20430        30.16
         bike day_type  hour  trips  missing_ids  missing_pct
 classic_bike  weekday     7   3242            0         0.00
 classic_bike  weekday     8   5931            0         0.00
 classic_bike  weekday     9   3566            0         0.00
 classic_bike  weekday    16   4306            0         0.00
 classic_bike  weekday    17   6328            0         0.00
 classic_bike  weekday    18   5150            0         0.00
electric_bike  weekday     7   6079         2203        36.24
electric_bike  weekday     8  11571         4471        38.64
electric_bike  weekday     9   7110         2806        39.47
electric_bike  weekday    16  12626         5433        43.03
electric_